# Linear Regression from scratch
(scratch: 긁어서 만든 달리기 출발선, 아무것도 없는 출발선 상태에서 linear regression을 하다)


# 구현할 것
- 공부시간과 성적간의 관계를 모델링한다.
    - **머신러닝 모델(모형)이란** 수집한 데이터를 기반으로 입력값(Feature)와 출력값(Target)간의 관계를 하나의 공식으로 정의한 함수이다. 그 공식을 찾는 과정을 **모델링**이라고 한다.
    - 이 예제에서는 공부한 시험시간으로 점수를 예측하는 모델을 정의한다.
    - 입력값과 출력값 간의 관계를 정의할 수있는 다양한 함수(공식)이 있다. 여기에서는 딥러닝과 관계가 있는 **Linear Regression** 을 사용해본다.

# 데이터 확인
- 입력데이터: 공부시간
- 출력데이터: 성적

|공부시간|점수|
|-|-|
|1|20|
|2|40|
|3|60|

우리가 수집한 공부시간과 점수 데이터를 바탕으로 둘 간의 관계를 식으로 정의 할 수 있으면 **내가 몇시간 공부하면 점수를 얼마 받을 수 있는지 예측할 수 있게 된다.**   
수집한 데이터를 기반으로 앞으로 예측할 수있는 모형을 만드는 것이 머신러닝 모델링이다.

  

## 학습(훈련) 데이터셋 만들기
- 모델을 학습시키기 위한 데이터셋을 구성한다.
- 입력데이터와 출력데이터을 각각 다른 행렬로 구성한다.
- 하나의 데이터 포인트의 입력/출력 값은 같은 index에 정의한다.

### 선형회귀 (Linear Regression)
- Feature들의 가중합을 이용해 Target을 추정한다.
- Feature에 곱해지는 가중치(weight)들은 각 Feature가 Target 얼마나 영향을 주는지 영향도가 된다.
    - 음수일 경우는 target값을 줄이고 양수일 경우는 target값을 늘린다.
    - 가중치가 0에 가까울 수록 target에 영향을 주지 않는 feature이고 0에서 멀수록 target에 많은 영향을 준다.
- 모델 학습과정에서 가장 적절한 Feature의 가중치를 찾아야 한다.
      

\begin{align}
&\large \hat{y} = W\cdot X + b\\
&\small \hat{y}: \text{모델추정값}\\
&\small W: \text{가중치}\\
&\small X: \text{Feature(입력값)}\\
&\small b: \text{bias(편향)}
\end{align}



## Train dataset 구성
- Train data는 feature(input)와 target(output) 각각 2개의 행렬로 구성한다.
- Feature의 행은 관측치(개별 데이터)를 열을 Feature(특성, 변수)를 표현한다. 이 문제에서는 `공부시간` 1개의 변수를 가진다.
- Target은 모델이 예측할 대상으로 행은 개별 관측치, 열은 각 항목에 대한 정답으로 구성한다.   
  이 문제에서 예측할 항목은 `시험점수` 한개이다.

In [1]:
import torch

X_train = torch.tensor([[1.], [2.], [3.]])
y_train = torch.tensor([[20.], [40.], [60.]])

X_train.size(), y_train.size()

import torch

X_train = torch.tensor([[1.], [2.], [3.]])
y_train = torch.tensor([[20.], [40.], [60.]])

X_train.size(), y_train.size()

(torch.Size([3, 1]), torch.Size([3, 1]))

## 파라미터 (weight, bias) 정의
- 학습대상/최적화 대상

In [2]:
weight = torch.randn(1, 1, requires_grad=True) # weight.shape은 [입력 피쳐수, 출력(정답) 피쳐수]
bias = torch.randn(1, requires_grad=True) # bias.shape은 [출력(정답) 피쳐수]
print(weight.size(), bias.size())

weight = torch.randn(1, 1, requires_grad=True)
bias = torch.randn(1, requires_grad=True)
print(weight.size(), bias.size())

# 내적의 의미? (3, 1) @ (1, 1) = (3, 1)
# 공부시간  국어 점수, 영어 점수, 수학 점수 (3, 1) @ (1, 3) = (3, 3) bias는 (3,)


torch.Size([1, 1]) torch.Size([1])
torch.Size([1, 1]) torch.Size([1])


In [3]:
weight

weight

tensor([[-0.4900]], requires_grad=True)

In [4]:
bias

bias

tensor([0.7696], requires_grad=True)

In [5]:
# 추론
pred = X_train @ weight + bias
print("추론한 시험점수:", pred)

pred = X_train @ weight + bias
print("추론한 시험점수:", pred)

추론한 시험점수: tensor([[ 0.2796],
        [-0.2104],
        [-0.7004]], grad_fn=<AddBackward0>)
추론한 시험점수: tensor([[ 0.2796],
        [-0.2104],
        [-0.7004]], grad_fn=<AddBackward0>)


In [6]:
pred.grad_fn # weight와 bias가 requires_grad = True라서 pred도 requires_grad = True가 된다
pred.grad_fn 

In [7]:
# 오차 계산(MSE)
loss = torch.mean((pred-y_train)**2, dim=0)
loss

tensor([1896.7690], grad_fn=<MeanBackward1>)

In [8]:
(pred-y_train)**2

tensor([[ 388.8940],
        [1616.8756],
        [3684.5374]], grad_fn=<PowBackward0>)

In [9]:
# weight bias를 업데이트
# gradient를 계산(loss/weight, loss/bias)
loss.backward()

In [10]:
weight.grad, bias.grad

(tensor([[-188.1616]]), tensor([-80.4208]))

In [11]:
# 2. weight bias 업데이트

lr = 0.12
weight.data = weight.data -lr*weight.grad
bias.data = bias.data -lr*bias.grad

print(weight.data, bias.data)

tensor([[22.0894]]) tensor([10.4201])


In [12]:
pred2 = X_train @ weight + bias
print(pred2) 

tensor([[32.5095],
        [54.5989],
        [76.6883]], grad_fn=<AddBackward0>)


In [13]:
# 새로운 MSE
torch.mean((pred2-y_train)**2)

tensor(216.0376, grad_fn=<MeanBackward0>)

In [14]:
#loss가 0이 될 때까지 반복 실행

### 모델링

In [15]:
# 모델 정의(함수 또는 클래스로 정의)
weight = torch.randn(1, 1, requires_grad=True)
bias = torch.randn(1, requires_grad=True)

def linear_model(X):
    return X @ weight + bias

    

In [16]:
# 오차계산(MSE)
def mse_loss(pred, y):
    return torch.mean((pred-y)**2)
    

### 학습
1. 모델을 이용해 추정한다.
   - pred = model(input)
1. loss를 계산한다.
   - loss = loss_fn(pred, target)
1. 계산된 loss를 파라미터에 대해 미분하여 계산한 gradient 값을 각 파라미터에 저장한다.
   - loss.backward()
1. optimizer를 이용해 파라미터를 update한다.
   - optimizer.step()  
1. 파라미터의 gradient(미분값)을 0으로 초기화한다. w=w-le* 라운드w/라운드 w)
   - optimizer.zero_grad()
- 위의 단계를 반복한다.   

In [17]:
epochs = 2000 
for epoch in range(epochs):
    # 1. 추론
    pred = linear_model(X_train)
    # 2. 오차
    loss = mse_loss(pred, y_train)
    # 3. 파라미터에 대한 그래디언트 계산
    loss.backward()
    # 4. 파라미터 업데이트(최적화)
    weight.data = weight.data -lr*weight.grad
    bias.data = bias.data -lr*bias.grad
    # 5. 파라미터 그래드 초기화
    weight.grad = None
    bias.grad = None

    if epoch % 100 ==0 or epoch ==(epochs-1): # 100번 반복과 마지막에 loss 출력
        print(f"[{epoch+1}/{epochs}]-loss:{loss.item()}")

# epochs pronunciation, meaning
# tensor has many character (ex, data, grad etc.)
# 1 - 5 phrase is one course of optimization


[1/2000]-loss:1992.43310546875
[101/2000]-loss:0.015221834182739258
[201/2000]-loss:4.364570486359298e-05
[301/2000]-loss:1.2483239686389425e-07
[401/2000]-loss:3.98964999925866e-10
[501/2000]-loss:9.701277108031814e-12
[601/2000]-loss:9.701277108031814e-12
[701/2000]-loss:9.701277108031814e-12
[801/2000]-loss:9.701277108031814e-12
[901/2000]-loss:9.701277108031814e-12
[1001/2000]-loss:9.701277108031814e-12
[1101/2000]-loss:9.701277108031814e-12
[1201/2000]-loss:9.701277108031814e-12
[1301/2000]-loss:9.701277108031814e-12
[1401/2000]-loss:9.701277108031814e-12
[1501/2000]-loss:9.701277108031814e-12
[1601/2000]-loss:9.701277108031814e-12
[1701/2000]-loss:9.701277108031814e-12
[1801/2000]-loss:9.701277108031814e-12
[1901/2000]-loss:9.701277108031814e-12
[2000/2000]-loss:9.701277108031814e-12


In [18]:
weight.data, bias.data

(tensor([[20.0000]]), tensor([1.0440e-05]))

In [19]:
with torch.no_grad():
    p = linear_model(X_train)
    print(p)

tensor([[20.0000],
        [40.0000],
        [60.0000]])


In [20]:
time = torch.tensor([[6.], [8.2]], dtype=torch.float32)
with torch.no_grad():
    p2 = linear_model(time)
    print(p2)

tensor([[120.0000],
        [164.0000]])


# 다중 입력, 다중 출력
- 다중입력: Feature가 여러개인 경우
- 다중출력: Output 결과가 여러개인 경우

다음 가상 데이터를 이용해 사과와 오렌지 수확량을 예측하는 선형회귀 모델을 정의한다.  
[참조](https://www.kaggle.com/code/aakashns/pytorch-basics-linear-regression-from-scratch)


|온도(F)|강수량(mm)|습도(%)|사과생산량(ton)|오렌지생산량|
|-|-|-|-:|-:|
|73|67|43|56|70|
|91|88|64|81|101|
|87|134|58|119|133|
|102|43|37|22|37|
|69|96|70|103|119|

```
사과수확량  = w11 * 온도 + w12 * 강수량 + w13 * 습도 + b1
오렌지수확량 = w21 * 온도 + w22 * 강수량 + w23 *습도 + b2
```

- `온도`, `강수량`, `습도` 값이 **사과**와, **오렌지 수확량**에 어느정도 영향을 주는지 가중치를 찾는다.
    - 모델은 사과의 수확량, 오렌지의 수확량 **두개의 예측결과를 출력**해야 한다.
    - 사과에 대해 예측하기 위한 weight 3개와 오렌지에 대해 예측하기 위한 weight 3개 이렇게 두 묶음, 총 6개의 weight를 정의하고 학습을 통해 가장 적당한 값을 찾는다.
        - `개별 과일를 예측하기 위한 weight들 @ feature들` 의 계산 결과를  **Node, Unit, Neuron** 이라고 한다.
        - 두 과일에 대한 Unit들을 묶어서 **Layer** 라고 한다.
- 목적은 우리가 수집한 train 데이터셋을 이용해 **정확한 예측을 위한 weight와 bias 들**을 찾는 것이다.

## Train Dataset
- Train data는 feature(input)와 target(output) 각각 2개의 행렬로 구성한다.
- Feature의 행은 관측치(개별 데이터)를 열을 Feature(특성, 변수)를 표현한다. 이 문제에서는 `온도, 강수량, 습도` 세개의 변수를 가진다.
- Target은 모델이 예측할 대상으로 행은 개별 관측치, 열은 각 항목에 대한 정답으로 구성한다. 이 문제에서 예측할 항목은 `사과수확량, 오렌지 수확량` 2개의 값이다.

In [21]:
#  input: 생산환경 (temp, rainfall, humidity) : (5, 3)
environs = [
    [73, 67, 43], 
    [91, 88, 64], 
    [87, 134, 58], 
    [102, 43, 37], 
    [69, 96, 70]
]

# Targets: 생산량 - (apples, oranges) - (5, 2)
apple_orange_output = [
    [56, 70], 
    [81, 101], 
    [119, 133], 
    [22, 37], 
    [103, 119]
]

# list -> torch transform

In [22]:
import torch
# Dataset을 torch.Tensor로 생성
X = torch.tensor(environs, dtype=torch.float32)
y = torch.tensor(apple_orange_output, dtype=torch.float32)
X.shape, y.shape

(torch.Size([5, 3]), torch.Size([5, 2]))

In [23]:
X

tensor([[ 73.,  67.,  43.],
        [ 91.,  88.,  64.],
        [ 87., 134.,  58.],
        [102.,  43.,  37.],
        [ 69.,  96.,  70.]])

## weight와 bias
- weight: 각 feature들이 생산량에 영향을 주었는지의 가중치로 feature에 곱해줄 값.
    - 사과, 오렌지의 생산량을 구해야 하므로 가중치가 두개가 된다.
    - weight의 shape: `(3, 2)`
- bias는 모든 feature들이 0일때 생산량이 얼마일지를 나타내는 값으로 feature와 weight간의 가중합 결과에 더해줄 값이다.
    - 사과, 오렌지의 생산량을 구하므로 bias가 두개가 된다.
    - bias의 shape: `(2, )`

### Linear Regression model
모델은 weights `w`와 inputs `x`의 내적(dot product)한 값에 bias `b`를 더하는 함수.

$$
\hspace{2.5cm} X \hspace{1.1cm} \cdot \hspace{1.2cm} W \hspace{1.2cm}  + \hspace{1cm} b \hspace{2cm}
$$

$$
\left[ \begin{array}{cc}
73 & 67 & 43 \\
91 & 88 & 64 \\
\vdots & \vdots & \vdots \\
69 & 96 & 70
\end{array} \right]
%
\cdot
%
\left[ \begin{array}{cc}
w_{11} & w_{21} \\
w_{12} & w_{22} \\
w_{13} & w_{23}
\end{array} \right]
%
+
%
\left[ \begin{array}{cc}
b_{1} & b_{2} \\
b_{1} & b_{2} \\
\vdots & \vdots \\
b_{1} & b_{2} \\
\end{array} \right]
$$


<center style="font-size:0.9em">
$w_{11},\,w_{12},\,w_{13}$: 사과 생산량 계산시 각 feature들(생산환경)에 곱할 가중치   <br>
$w_{21},\,w_{22},\,w_{23}$: 오렌지 생산량 계산시 각 feature들(생산환경)에 곱할 가중치    
</center>

<center>
<img src="https://raw.githubusercontent.com/kgmyhGit/image_resource/main/deeplearning/figures/3_unit_layer.png">
</center>

In [ ]:
# modelling
# weight shape (3, feature no., 2: output no.)
# bias shape(2: output no.)
weight = torch.randn(3, 2, requires_grad=True)
bias = torch.randn(2, requires_grad=True)
def model(X):
    return X @ weight + bias

In [30]:
y_hat = model(X)
print(y_hat.shape)
y_hat

torch.Size([5, 2])


tensor([[ -42.9501,   51.0425],
        [ -52.9742,   58.4229],
        [  11.2862,   44.5810],
        [-109.1800,   92.6404],
        [ -15.8187,   28.8738]], grad_fn=<AddBackward0>)

In [31]:
y # answer

tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.]])

In [32]:
# loss func defnition
def loss_fn(pred, y):
    """mse"""
    return torch.mean((pred - y)**2)



In [ ]:
# learning: using train set, optimization of weight & bias
epochs = 5000
lr = 0.00001
for epoch in range(epochs):
    # 1. using model prediction
    pred = model(X)
    # 2. mse caculation
    loss = loss_fn(pred, y)
    # 3. gradient of weight & bias   
    loss.backward()
    # 4. weight & bias optimization
    weight.data = weight.data - lr*weight.grad
    bias.data = bias.data - lr*bias.grad
    # 5. initializing
    weight.grad = None
    bias.grad = None

    # log output
    if epoch % 100 == 0 or epoch == (epochs - 1): # 100 epoch
        print(f"[{epoch}/{epochs}] - loss: {loss.item(): .5f}") # .5f ?


[0/5000] - loss:  9187.72461
[100/5000] - loss:  471.58408
[200/5000] - loss:  180.32932
[300/5000] - loss:  90.48576
[400/5000] - loss:  58.54261
[500/5000] - loss:  43.99372
[600/5000] - loss:  35.27742
[700/5000] - loss:  28.98732
[800/5000] - loss:  24.03300
[900/5000] - loss:  19.99685
[1000/5000] - loss:  16.66917
[1100/5000] - loss:  13.91425
[1200/5000] - loss:  11.63033
[1300/5000] - loss:  9.73600
[1400/5000] - loss:  8.16456
[1500/5000] - loss:  6.86085
[1600/5000] - loss:  5.77930
[1700/5000] - loss:  4.88200
[1800/5000] - loss:  4.13760
[1900/5000] - loss:  3.52001
[2000/5000] - loss:  3.00764
[2100/5000] - loss:  2.58258
[2200/5000] - loss:  2.22993
[2300/5000] - loss:  1.93736
[2400/5000] - loss:  1.69465
[2500/5000] - loss:  1.49327
[2600/5000] - loss:  1.32622
[2700/5000] - loss:  1.18763
[2800/5000] - loss:  1.07264
[2900/5000] - loss:  0.97724
[3000/5000] - loss:  0.89809
[3100/5000] - loss:  0.83245
[3200/5000] - loss:  0.77797
[3300/5000] - loss:  0.73278
[3400/500

##  모델링

In [36]:
print(weight)
print("-------")
print(bias)

tensor([[-0.3873, -0.3035],
        [ 0.8512,  0.7982],
        [ 0.6882,  0.8980]], requires_grad=True)
-------
tensor([-1.4127,  0.4605], requires_grad=True)


In [39]:
# inference using model
with torch.no_grad():
    p = model(X)
    print(p)

tensor([[ 56.9361,  70.3946],
        [ 82.2914, 100.5509],
        [118.8645, 133.0938],
        [ 21.1486,  37.0470],
        [101.7496, 119.0025]])


In [38]:
y

tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.]])

In [ ]:
new_X = [[63.2, 121.6, 32.1]]
new_X = torch.tensor(new_X, dtype = torch.float32)
with torch.no_grad():
    new_pred = model(new_X)
    print(new_pred)


print("apple prediction production:", new_pred[0,0].item())
print("orange prediction production:", new_pred[0,1].item())

tensor([[ 99.7032, 107.1623]])
apple prediction production: 99.70319366455078
orange prediction production: 107.16228485107422


# pytorch built-in 모델을 사용해 Linear Regression 구현

In [42]:
inputs = torch.tensor(
    [[73, 67, 43], 
     [91, 88, 64], 
     [87, 134, 58], 
     [102, 43, 37], 
     [69, 96, 70]], dtype=torch.float32)

In [43]:
targets = torch.tensor(
    [[56, 70], 
    [81, 101], 
    [119, 133], 
    [22, 37], 
    [103, 119]], dtype=torch.float32)

## torch.nn.Linear
Pytorch는 torch.nn.Linear 클래스를 통해 Linear Regression 모델을 제공한다.  
torch.nn.Linear에 입력 feature의 개수와 출력 값의 개수를 지정하면 random 값으로 초기화한 weight와 bias들을 생성해 모델을 구성한다.
- `torch.nn.Linear(input feature의 개수 , output 값의 개수)`# nn: neural network

## Optimizer와 Loss 함수 정의
- **Optimizer**: 계산된 gradient값을 이용해 파라미터들을 업데이트 하는 함수 # 4th stage
- **Loss 함수**: 정답과 모델이 예측한 값사이의 차이(오차)를 계산하는 함수. # 2nd stage
  - 모델을 최적화하는 것은 이 함수의 값을 최소화하는 것을 말한다. 
- `torch.optim` 모듈에 다양한 Optimizer 클래스가 구현되어 있다.
- `torch.nn` 또는 `torch.nn.functional` 모듈에 다양한 Loss 함수가 제공된다. 

In [44]:
inputs.shape, targets.shape

(torch.Size([5, 3]), torch.Size([5, 2]))

In [46]:
# 선형회귀 모델을 정의. torch.nn.Linear 클래스
import torch
import torch.nn as nn

model = nn.Linear(3, 2)  # 3: input feature 개수, 2: output 수

In [47]:
# loss 함수
loss_fn = torch.nn.MSELoss()  # 클래스
# loss_fn = torch.nn.functional.mse_loss # 함수 
# class & function difference?


In [54]:
model.parameters()
print(model.parameters())
list(model.parameters())

<generator object Module.parameters at 0x000002120BDCCE40>


[Parameter containing:
 tensor([[-0.0344,  0.3824, -0.3441],
         [ 0.0540, -0.5296,  0.3287]], requires_grad=True),
 Parameter containing:
 tensor([0.4857, 0.5541], requires_grad=True)]

In [57]:
# optimizer (torch.optim 모듈에 정의): weight.data = weight.data - lr * weight.grad
optimizer = torch.optim.SGD(
    model.parameters(), # 최적화 대상 파라미터들을 model에서 조회해서 전달.
    lr = 0.00001,       # Learning Rage
) # stochastic gradient dscent

In [58]:
list(model.parameters())

[Parameter containing:
 tensor([[-0.0344,  0.3824, -0.3441],
         [ 0.0540, -0.5296,  0.3287]], requires_grad=True),
 Parameter containing:
 tensor([0.4857, 0.5541], requires_grad=True)]

## Model Train

In [77]:
epochs = 5000

for epoch in range(epochs):
    # 추론
    pred = model(inputs)  # callable type
    # loss 계산
    loss = loss_fn(pred, targets) # torch.nn.functional.mse_loss(pred, targets) # (모델추정값, 정답)
    # gradient 계산
    loss.backward()
    # 파라미터 업데이트: optimizer.step()
    optimizer.step()
    # 파라미터 초기화 w.grad=None, b.grad=None
    optimizer.zero_grad()
    # 현재 epoch 학습 결과를 log로 출력
    if epoch % 100 == 0 or epoch == epochs-1:
        print(f"[{epoch+1:04d}/{epochs}] - {loss.item()}") # :04d 04 digit d int

[0001/5000] - 10104.845703125
[0101/5000] - 191.9414825439453
[0201/5000] - 69.87418365478516
[0301/5000] - 32.958045959472656
[0401/5000] - 20.389774322509766
[0501/5000] - 15.030156135559082
[0601/5000] - 12.005590438842773
[0701/5000] - 9.895343780517578
[0801/5000] - 8.256608963012695
[0901/5000] - 6.928458213806152
[1001/5000] - 5.835428714752197
[1101/5000] - 4.931084632873535
[1201/5000] - 4.181522369384766
[1301/5000] - 3.5598533153533936
[1401/5000] - 3.0441606044769287
[1501/5000] - 2.616349220275879
[1601/5000] - 2.2614238262176514
[1701/5000] - 1.9669612646102905
[1801/5000] - 1.7226784229278564
[1901/5000] - 1.5200165510177612
[2001/5000] - 1.3518779277801514
[2101/5000] - 1.2123866081237793
[2201/5000] - 1.0966572761535645
[2301/5000] - 1.0006532669067383
[2401/5000] - 0.920993983745575
[2501/5000] - 0.8549168705940247
[2601/5000] - 0.8000979423522949
[2701/5000] - 0.7546125650405884
[2801/5000] - 0.7168788313865662
[2901/5000] - 0.685566782951355
[3001/5000] - 0.65960389

In [78]:
# callabe 
class Test:
    def __call__(self, X, y):
        return X+y
t = Test()
t(10, 20)
# model: using class like function

30

In [79]:
# 추론 => gradient 계산을 할 필요가 없다. ==> grad_fn을 만들 필요가 없다. 그래서 torch.no_grad() 블록에서 추론 작업을 실행한다.
with torch.no_grad():
    pred = model(inputs)

In [80]:
targets

tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.]])

In [81]:
pred

tensor([[ 57.2533,  70.4027],
        [ 82.1016, 100.6040],
        [118.7923, 132.9603],
        [ 21.1027,  37.0083],
        [101.8211, 119.1358]])

In [82]:
loss_fn(pred, targets)

tensor(0.5361)

In [83]:
# 학습 로직을 함수 구현
def train(inputs, targets, epochs, model, loss_fn, optimizer):

    for epoch in range(epochs):
        # 추론
        pred = model(inputs)
        # loss 계산
        loss = loss_fn(pred, targets) # torch.nn.functional.mse_loss(pred, targets) # (모델추정값, 정답)
        # gradient 계산
        loss.backward()
        # 파라미터 업데이트: optimizer.step()
        optimizer.step()
        # 파라미터 초기화 w.grad=None, b.grad=None
        optimizer.zero_grad()
        # 현재 epoch 학습 결과를 log로 출력
        if epoch % 100 == 0 or epoch == epochs-1:
            print(f"[{epoch+1:04d}/{epochs}] - {loss.item()}")

In [84]:
model = nn.Linear(3, 2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)

In [85]:
train(inputs, targets, 5000, model, nn.functional.mse_loss, optimizer)

[0001/5000] - 18473.462890625
[0101/5000] - 3.6498475074768066
[0201/5000] - 0.9973465800285339
[0301/5000] - 0.5910183191299438
[0401/5000] - 0.5287443399429321
[0501/5000] - 0.5191970467567444
[0601/5000] - 0.5177274942398071
[0701/5000] - 0.5174962282180786
[0801/5000] - 0.5174574255943298
[0901/5000] - 0.5174420475959778
[1001/5000] - 0.5174368023872375
[1101/5000] - 0.5174347758293152
[1201/5000] - 0.517431378364563
[1301/5000] - 0.5174216628074646
[1401/5000] - 0.517417311668396
[1501/5000] - 0.5174115896224976
[1601/5000] - 0.5174084901809692
[1701/5000] - 0.5174010396003723
[1801/5000] - 0.5173938870429993
[1901/5000] - 0.5173892378807068
[2001/5000] - 0.5173847079277039
[2101/5000] - 0.5173763632774353
[2201/5000] - 0.5173742771148682
[2301/5000] - 0.517367422580719
[2401/5000] - 0.5173622369766235
[2501/5000] - 0.5173572301864624
[2601/5000] - 0.5173481106758118
[2701/5000] - 0.517345130443573
[2801/5000] - 0.5173378586769104
[2901/5000] - 0.5173331499099731
[3001/5000] - 0.5

In [86]:
model(inputs)

tensor([[ 57.1365,  70.3534],
        [ 82.2251, 100.6425],
        [118.6972, 132.9508],
        [ 21.0850,  37.0096],
        [101.9163, 119.1454]], grad_fn=<AddmmBackward0>)

In [ ]:
with torch.no_grad():
    p = model(inputs)
    print(p)
# grad_fn=<AddmmBackward0> no exists

tensor([[ 57.1365,  70.3534],
        [ 82.2251, 100.6425],
        [118.6972, 132.9508],
        [ 21.0850,  37.0096],
        [101.9163, 119.1454]])
